# Intacart Reorder Prediction

Setup some constants

In [1]:
from pathlib import Path

CASE_DIR = Path("instacart-market-basket-analysis")

In [2]:
import subprocess
import sys
import pandas as pd

# Ensure the data exists, if not download
if not CASE_DIR.exists():
    subprocess.run([sys.executable, "load_data.py"], check=True)

AISLES_DF = pd.read_csv(CASE_DIR / "aisles.csv")
DEPARTMENTS_DF = pd.read_csv(CASE_DIR / "departments.csv")
ORDERS_DF = pd.read_csv(CASE_DIR / "orders.csv")
PRIOR_DF = pd.read_csv(CASE_DIR / "order_products__prior.csv")
TRAIN_DF = pd.read_csv(CASE_DIR / "order_products__train.csv")
PRODUCTS_DF = pd.read_csv(CASE_DIR / "products.csv")

### Build a Product x User Table that will be used to train a predictor for the next purchase of that user
Prior orders are history of purchases. User bought Product at some point

In [3]:
prior_orders = ORDERS_DF[ORDERS_DF.eval_set == "prior"]
history = PRIOR_DF.merge(prior_orders[["order_id", "user_id", "order_number"]], on="order_id")
n_orders = prior_orders.groupby("user_id").order_number.max().rename("n_all_orders")

# This is User x Product
feature = (
    history.groupby(["user_id", "product_id"])
        .agg(up_n_orders=("order_id", "size"),
        up_last_order=("order_number", "max"),
        up_avg_cart_pos=("add_to_cart_order", "mean"))
    .reset_index()
    .merge(n_orders, on="user_id")
)
feature["up_order_ratio"] = feature.up_n_orders / feature.n_all_orders
feature["up_orders_since_last"] = feature.n_all_orders - feature.up_last_order

Train orders are last order.  
The label is True if the User+Product pair exists in the train set.  
User bought product in last order.
Data has all the combinations of User+Product

In [4]:
target_orders = ORDERS_DF[ORDERS_DF.eval_set == "train"][
    ["order_id", "user_id", "order_dow","order_hour_of_day", "days_since_prior_order"]]
label = (TRAIN_DF.merge(target_orders[["order_id", "user_id"]], on="order_id")
    [["user_id", "product_id"]].assign(label=1))
data = (
    feature.merge(target_orders.drop(columns="order_id"), on="user_id")
        .merge(label, on=["user_id", "product_id"], how="left")
)
data["label"] = data.label.fillna(0).astype("int8")

### Train Desition Tree Classifier on Train Data
Feature columns predict label (history predicts last purchase)

In [5]:
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit

features = ["up_n_orders", "up_order_ratio", "up_orders_since_last", "up_avg_cart_pos",
    "n_all_orders", "order_dow", "order_hour_of_day", "days_since_prior_order"]

split = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, va_idx = next(split.split(data, groups=data.user_id))
tr, va = data.iloc[tr_idx], data.iloc[va_idx]

model = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=100,
    colsample_bytree=0.8
)

model.fit(tr[features], tr.label,
          eval_X=va[features], eval_y=va.label,
          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])

va_prob = model.predict_proba(va[features])[:, 1]

print(pd.Series(model.feature_importances_, index=features).sort_values(ascending=False))

[LightGBM] [Info] Number of positive: 661978, number of negative: 6123547
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040754 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 826
[LightGBM] [Info] Number of data points in the train set: 6785525, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.097557 -> initscore=-2.224664
[LightGBM] [Info] Start training from score -2.224664
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.253509
[200]	valid_0's binary_logloss: 0.253439
Early stopping, best iteration is:
[166]	valid_0's binary_logloss: 0.253432
n_all_orders              1697
days_since_prior_order    1695
up_order_ratio            1687
up_orders_since_last      1264
up_avg_cart_pos           1214
order_hour_of_day         1046
up_n_orders                956
ord

In [6]:
imp = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=features)
print((imp / imp.sum()).sort_values(ascending=False).round(3))

up_orders_since_last      0.431
up_order_ratio            0.386
up_n_orders               0.111
days_since_prior_order    0.028
n_all_orders              0.024
up_avg_cart_pos           0.016
order_hour_of_day         0.002
order_dow                 0.002
dtype: float64


Feature Importances show how strong a column is a predictor of purchase.
up_orders_since_last being the strongest mean there's bias for recency in the purchase desition.
Followed by up_order_ratio shows habit.

hour of day, day of week, and cart position being the weakest show these may dont matter, are candidates to be exluded.

### Let's proceed with validation for now

In [7]:
def f1_score(actual: set, predicted: set) -> float:
    if not actual and not predicted:
        return 1.0
    tp = len(actual & predicted)
    if tp == 0:
        return 0.0
    p, r = tp / len(predicted), tp / len(actual)
    return 2 * p * r / (p + r)

def mean_f1(df, mask):
    actual = df[df.label == 1].groupby("user_id").product_id.apply(set)
    pred = df[mask].groupby("user_id").product_id.apply(set)
    return np.mean([f1_score(actual.get(u, set()), pred.get(u, set())) for u in df.user_id.unique()])

In [8]:
import numpy as np

va = va.assign(prob=va_prob)

prob_threshold = .3
print(f"prob > {prob_threshold}\t\t:", mean_f1(va, va.prob > prob_threshold))
print("repeat last order\t:", mean_f1(va, va.up_orders_since_last == 0))
print("ratio >= 0.5\t\t:", mean_f1(va, va.up_order_ratio >= 0.5))

prob > 0.3		: 0.3427631215877016
repeat last order	: 0.31294634068219923
ratio >= 0.5		: 0.2843639041368833


# Predict Test Orders

In [9]:
# Retrain model in full Train Set

final_m = lgb.LGBMClassifier(
    n_estimators=model.best_iteration_,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=100,
    colsample_bytree=0.8)
final_m.fit(data[features], data.label)

[LightGBM] [Info] Number of positive: 828824, number of negative: 7645837
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.041427 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 826
[LightGBM] [Info] Number of data points in the train set: 8474661, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.097800 -> initscore=-2.221909
[LightGBM] [Info] Start training from score -2.221909


,num_leaves,63
,learning_rate,0.05
,n_estimators,166
,min_child_samples,100
,colsample_bytree,0.8
,boosting_type,'gbdt'
,max_depth,-1
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0


In [10]:
test_orders = ORDERS_DF[ORDERS_DF.eval_set == "test"][
    ["order_id", "user_id", "order_dow", "order_hour_of_day", "days_since_prior_order"]]

test = feature.merge(test_orders, on="user_id")
test["prob"] = final_m.predict_proba(test[features])[:, 1]

pred = (
    test[test.prob > prob_threshold]
        .groupby("order_id").product_id
        .apply(lambda s: " ".join(map(str, s)))
        .rename("products").reset_index()
)

sub = test_orders[["order_id"]].merge(pred, on="order_id", how="left")
sub["products"] = sub.products.fillna("None")
sub

,order_id,products
0,2774568,17668 18599 21903 39190 47766
1,329954,None
2,1528013,21903 38293
3,1376945,8309 14947 27959
4,1356845,7076 10863 13176 14992
...,...,...
74995,2728930,2693 17038 24852
74996,350108,5646 10199 15649 30561 43961 49075
74997,1043943,11520 42623
74998,2821651,3397 11068 13176 20995 27966 33754 33787 36011...
